# Stored execution evidence — TGS Salt Identification Challenge

This public evidence copy preserves every textual output cell supplied in `agent_final_submission.zip`.
The original notebook SHA-256 is `6dd5efc5a66a9851db0177502252a6e20ebd1945dcfecbc6f7aa3aeb069313f3`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Jiaozi Agent — TGS Salt Identification Challenge

This notebook preserves the original Jiaozi Module 1 → 4 workflow and adapts it to the
**TGS Salt Identification Challenge**, a binary semantic-segmentation benchmark.

The active flow is:

1. Module 1 parses the TGS task and constraints with the configured LLM.
2. Module 2 analyzes the official 101×101 seismic images and salt masks.
3. Module 3 ranks segmentation recipes.
4. Module 4 generates the auditable Jiaozi project and configuration.
5. A TGS real-data execution adapter trains the first-ranked recipe on real masks,
   selects the checkpoint with the official competition metric, and produces the
   required column-major RLE `submission.csv`.

The adapter is required because this Jiaozi branch currently generates real local
dataloaders only for classification/feature extraction; its stock segmentation path
otherwise falls back to synthetic data.


## Before running

1. In Colab **Secrets**, add `OPENAI_API_KEY` and `KAGGLE_API_TOKEN`.
2. Confirm your Kaggle account is eligible to access this ended competition. Accepting
   rules is necessary but does not guarantee that a late submission is available.
3. Select a GPU runtime (T4 or L4).
4. Run the cells in order. Google Drive is used for resumable checkpoints.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

def normalize_repo_url(value: str) -> str:
    value = (value or '').strip()
    markdown_match = re.fullmatch(r'\[(.*?)\]\((https?://[^)]+)\)', value)
    if markdown_match:
        return markdown_match.group(2)
    plain_url_match = re.search(r'https?://\S+', value)
    if plain_url_match:
        return plain_url_match.group(0)
    return value

REPO_URL = normalize_repo_url(os.getenv('JIAOZI_REPO_URL', 'https://github.com/Isso-W/Jiaozi.git'))
REPO_REF = os.getenv('JIAOZI_REPO_REF', 'main')
REPO_DIR = Path('/content/Jiaozi')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

print("REPO_URL =", REPO_URL)
print("REPO_REF =", REPO_REF)

clone_cmd = [
    'git', 'clone',
    '--depth', '1',
    '--branch', REPO_REF,
    REPO_URL,
    str(REPO_DIR),
]

completed = subprocess.run(
    clone_cmd,
    capture_output=True,
    text=True,
)

print("=== git stdout ===")
print(completed.stdout)
print("=== git stderr ===")
print(completed.stderr)

completed.check_returncode()

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("\n=== requirements.txt ===")
print(Path("requirements.txt").read_text()[:4000])

print("\n=== installing requirements ===")
pip_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
    capture_output=True,
    text=True,
)

print(pip_result.stdout)
print(pip_result.stderr)
print("pip return code =", pip_result.returncode)

if pip_result.returncode != 0:
    raise RuntimeError("pip install failed. Check the pip output above.")

timm_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "timm>=1.0.20"],
    capture_output=True,
    text=True,
)
print(timm_result.stdout)
print(timm_result.stderr)
if timm_result.returncode != 0:
    raise RuntimeError("timm installation failed")

print('Jiaozi repository:', REPO_DIR)
print('Jiaozi repo URL:', REPO_URL)
print('Jiaozi ref:', REPO_REF)

pipeline_candidates = list(REPO_DIR.rglob("pipeline.py"))
print("pipeline.py candidates:")
for p in pipeline_candidates:
    print(" -", p)


REPO_URL = https://github.com/muzhi-hac/Jiaozi-rag-wang.git
REPO_REF = rag_wang
=== git stdout ===

=== git stderr ===
Cloning into '/content/Jiaozi'...


=== requirements.txt ===
# Runtime dependencies for the Jiaozi CV Auto-DL pipeline.
# Install with the same Python interpreter used to run tests, for example:
#   /usr/bin/python3 -m pip install -r requirements.txt

chromadb
datasets
networkx
numpy
openai>=2.0.0
pandas
pillow
scikit-learn
sentence-transformers
torch
torchvision
transformers

# Test runner for the module4_agent pytest suite.
pytest


=== installing requirements ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 175.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 190.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 185.0 MB/s eta 0:00:00
   ━━━━━━━━

In [2]:
import os
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_colab_secret(name: str) -> str:
    if userdata is None:
        return ""

    try:
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""


# 读取千问 API Key
qwen_api_key = read_colab_secret(
    "JIAOZI_DASHSCOPE_API_KEY"
)

if not qwen_api_key:
    raise RuntimeError(
        "Colab Secret 中缺少 "
        "JIAOZI_DASHSCOPE_API_KEY。"
    )

if not qwen_api_key.startswith("sk-"):
    raise RuntimeError(
        "JIAOZI_DASHSCOPE_API_KEY 格式不正确："
        "它应当是以 sk- 开头的 API Key。"
    )


# 读取百炼 OpenAI-compatible 服务地址
qwen_base_url = read_colab_secret(
    "DASHSCOPE_BASE_URL"
)

if not qwen_base_url:
    raise RuntimeError(
        "Colab Secret 中缺少 DASHSCOPE_BASE_URL。"
        "请填写百炼页面顶部的 OpenAI compatible 地址。"
    )

if qwen_base_url.startswith("sk-"):
    raise RuntimeError(
        "DASHSCOPE_BASE_URL 中误填了 API Key。"
        "这里应填写以 https:// 开头的服务地址。"
    )

if not qwen_base_url.startswith("https://"):
    raise RuntimeError(
        "DASHSCOPE_BASE_URL 格式不正确："
        "必须以 https:// 开头。"
    )

if not qwen_base_url.rstrip("/").endswith(
    "/compatible-mode/v1"
):
    raise RuntimeError(
        "DASHSCOPE_BASE_URL 看起来不是"
        "百炼 OpenAI-compatible 地址。"
    )


# 配置 Jiaozi 使用千问
os.environ[
    "JIAOZI_DASHSCOPE_API_KEY"
] = qwen_api_key

os.environ[
    "DASHSCOPE_BASE_URL"
] = qwen_base_url.rstrip("/")

os.environ[
    "JIAOZI_LLM_PROVIDER"
] = "qwen"

os.environ[
    "M1_LLM_PROVIDER"
] = "qwen"

os.environ[
    "M4_LLM_PROVIDER"
] = "qwen"

os.environ[
    "M1_QWEN_MODEL"
] = "qwen3.7-plus"

os.environ[
    "M4_QWEN_MODEL"
] = "qwen3.7-plus"


# 可选：读取 Kaggle Token
kaggle_token = read_colab_secret(
    "KAGGLE_API_TOKEN"
)

if kaggle_token:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    token_path = kaggle_dir / "access_token"
    token_path.write_text(
        kaggle_token,
        encoding="utf-8",
    )
    token_path.chmod(0o600)

    os.environ[
        "KAGGLE_API_TOKEN"
    ] = kaggle_token


# 只打印配置状态，不打印任何敏感值
print("LLM provider: qwen")
print("Qwen model: qwen3.7-plus")
print("Qwen API Key configured: True")
print("DashScope Base URL configured: True")
print(
    "Kaggle token configured:",
    bool(kaggle_token),
)

# 删除当前 Cell 中保存的局部变量
del qwen_api_key
del qwen_base_url
del kaggle_token


LLM provider: qwen
Qwen model: qwen3.7-plus
Qwen API Key configured: True
DashScope Base URL configured: True
Kaggle token configured: True


## Download the official TGS Salt data

The competition contains 4,000 labeled 101×101 images/masks and roughly 18,000 test
images. The extraction cell handles both the direct archive layout and Kaggle's nested
`competition_data.zip` layout.


In [3]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

def run_pip(args):
    print("Running:", " ".join([sys.executable, "-m", "pip"] + args))
    result = subprocess.run(
        [sys.executable, "-m", "pip"] + args,
        capture_output=True,
        text=True,
    )
    print("=== pip stdout ===")
    print(result.stdout)
    print("=== pip stderr ===")
    print(result.stderr)
    print("return code =", result.returncode)
    return result

# 先升级基础安装工具
run_pip(["install", "--upgrade", "pip", "setuptools", "wheel"])

# 再安装 Kaggle CLI，不要 -q
kaggle_install = run_pip(["install", "--upgrade", "kaggle"])

if kaggle_install.returncode != 0:
    raise RuntimeError("Kaggle CLI install failed. Check pip output above.")

# 从 Colab Secrets 读取 Kaggle access token
kaggle_token = userdata.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    raise RuntimeError("Please add KAGGLE_API_TOKEN in Colab Secrets.")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

access_token_path = kaggle_dir / "access_token"
access_token_path.write_text(kaggle_token.strip(), encoding="utf-8")
access_token_path.chmod(0o600)

os.environ["KAGGLE_API_TOKEN"] = kaggle_token.strip()

print("Kaggle access token configured.")

# 检查 kaggle 是否可用
subprocess.run(["kaggle", "--version"], check=False)


Running: /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
=== pip stdout ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.6 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

=== pip stderr ===
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.

return code = 0
Running: /usr/bin/python3 -m pip install --upgrade kaggle
=== pip stdout

CompletedProcess(args=['kaggle', '--version'], returncode=0)

In [4]:
KAGGLE_COMPETITION = "tgs-salt-identification-challenge"
!kaggle competitions download -c {KAGGLE_COMPETITION} -p /content/data


100% 445M/445M [00:14<00:00, 32.2MB/s]



In [5]:
from pathlib import Path
import zipfile

DATA_ROOT = Path("/content/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
KAGGLE_DATA_DIR = DATA_ROOT / KAGGLE_COMPETITION
KAGGLE_DATA_DIR.mkdir(parents=True, exist_ok=True)

outer_candidates = sorted(DATA_ROOT.glob(f"{KAGGLE_COMPETITION}*.zip"))
if not outer_candidates:
    raise FileNotFoundError(
        f"No downloaded archive matching {KAGGLE_COMPETITION}*.zip under {DATA_ROOT}"
    )

outer_zip = outer_candidates[0]
print("Extracting outer archive:", outer_zip, "->", KAGGLE_DATA_DIR)
with zipfile.ZipFile(outer_zip, "r") as archive:
    archive.extractall(KAGGLE_DATA_DIR)

# Extract every nested archive once. This covers both competition_data.zip and
# older train.zip/test.zip packaging without assuming one fixed Kaggle layout.
extracted = set()
while True:
    pending = [path for path in KAGGLE_DATA_DIR.rglob("*.zip") if path not in extracted]
    if not pending:
        break
    for nested_zip in sorted(pending):
        nested_target = nested_zip.with_suffix("")
        nested_target.mkdir(parents=True, exist_ok=True)
        print("Extracting nested:", nested_zip, "->", nested_target)
        with zipfile.ZipFile(nested_zip, "r") as archive:
            archive.extractall(nested_target)
        extracted.add(nested_zip)

print("KAGGLE_DATA_DIR =", KAGGLE_DATA_DIR)
print("CSV files:", [str(path.relative_to(KAGGLE_DATA_DIR)) for path in KAGGLE_DATA_DIR.rglob("*.csv")])
print("PNG count:", sum(1 for _ in KAGGLE_DATA_DIR.rglob("*.png")))


Extracting outer archive: /content/data/tgs-salt-identification-challenge.zip -> /content/data/tgs-salt-identification-challenge
Extracting nested: /content/data/tgs-salt-identification-challenge/competition_data.zip -> /content/data/tgs-salt-identification-challenge/competition_data
Extracting nested: /content/data/tgs-salt-identification-challenge/flamingo.zip -> /content/data/tgs-salt-identification-challenge/flamingo
Extracting nested: /content/data/tgs-salt-identification-challenge/test.zip -> /content/data/tgs-salt-identification-challenge/test
Extracting nested: /content/data/tgs-salt-identification-challenge/train.zip -> /content/data/tgs-salt-identification-challenge/train
KAGGLE_DATA_DIR = /content/data/tgs-salt-identification-challenge
CSV files: ['sample_submission.csv', 'depths.csv', 'train.csv', 'competition_data/competition_data/sample_submission.csv', 'competition_data/competition_data/depths.csv', 'competition_data/competition_data/train.csv', 'competition_data/__MACOS

## Mount Google Drive for caches and checkpoints

HuggingFace caches and training checkpoints can live on Drive so a recycled Colab runtime does not force every artifact to be downloaded or trained from scratch.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Everything HuggingFace caches (datasets + hub) lives here, on Drive.
HF_CACHE_DIR = '/content/drive/MyDrive/Jiaozi/hf_cache'
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = os.path.join(HF_CACHE_DIR, 'datasets')

print('HF_HOME =', os.environ['HF_HOME'])
print('First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).')
print('Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.')


Mounted at /content/drive
HF_HOME = /content/drive/MyDrive/Jiaozi/hf_cache
First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).
Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.


## Run the integrated Jiaozi pipeline

This cell keeps the original Module 1–4 sequence but supplies segmentation-aware local
data to Module 2. It verifies one-to-one image/mask pairs, computes salt coverage classes,
forces the official task/metric into the merged request, and injects only dataset paths
and benchmark facts into every generated candidate. Model, loss, optimizer, learning rate,
augmentation, finetuning strategy, and epoch recommendation remain Jiaozi outputs.


In [7]:
# TGS Salt Kaggle data -> Jiaozi Module 1-4

import json
import os
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

REPO_DIR = Path("/content/Jiaozi")
PIPELINE_PATH = REPO_DIR / "pipeline.py"
if not PIPELINE_PATH.exists():
    raise FileNotFoundError(f"No pipeline.py found at {PIPELINE_PATH}; rerun the clone cell.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

QUERY = (
    "Binary semantic segmentation for the Kaggle TGS Salt Identification Challenge. "
    "Each grayscale seismic image is 101 by 101 pixels and each mask marks salt versus sediment. "
    "The official primary metric is mean per-image precision over IoU thresholds "
    "0.50, 0.55, ..., 0.95; checkpoint selection and threshold tuning must use this metric. "
    "The foreground is imbalanced and empty masks are valid. Use an encoder-decoder segmentation "
    "model, paired image-mask augmentation, and a loss appropriate for binary imbalanced masks. "
    "Resource constraint: one Colab T4 or L4 GPU. Use only public, non-gated pretrained weights. "
    "The agent must select the recipe, optimizer, learning rate, augmentation, finetuning strategy, "
    "and recommended number of epochs. Preserve spatial output and predict one foreground logit."
)

REAL_TRAINING = True
EPOCHS = None  # None = use Jiaozi's recommended_epochs
OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")
DATASET_DIR = Path(KAGGLE_DATA_DIR)

def choose_csv(name):
    matches = sorted(DATASET_DIR.rglob(name), key=lambda path: (len(path.parts), str(path)))
    if not matches:
        raise FileNotFoundError(f"Could not find {name} under {DATASET_DIR}")
    return matches[0]

sample_path = choose_csv("sample_submission.csv")
depths_path = choose_csv("depths.csv")
sample = pd.read_csv(sample_path)
if list(sample.columns) != ["id", "rle_mask"]:
    raise ValueError(f"Unexpected sample_submission columns: {list(sample.columns)}")

mask_dirs = []
for directory in DATASET_DIR.rglob("masks"):
    count = sum(1 for _ in directory.glob("*.png")) if directory.is_dir() else 0
    if count:
        mask_dirs.append((count, directory))
if not mask_dirs:
    raise FileNotFoundError(f"No train/masks directory with PNG files under {DATASET_DIR}")
_, train_mask_dir = max(mask_dirs, key=lambda item: item[0])
train_image_dir = train_mask_dir.parent / "images"
if not train_image_dir.is_dir():
    raise FileNotFoundError(f"Expected sibling train images directory: {train_image_dir}")

train_ids = sorted(
    {path.stem for path in train_image_dir.glob("*.png")}
    & {path.stem for path in train_mask_dir.glob("*.png")}
)
if not train_ids:
    raise RuntimeError("No paired TGS training images and masks were found")

test_ids = sample["id"].astype(str).tolist()
test_id_set = set(test_ids)
image_dirs = []
for directory in DATASET_DIR.rglob("images"):
    if not directory.is_dir() or directory.resolve() == train_image_dir.resolve():
        continue
    stems = {path.stem for path in directory.glob("*.png")}
    overlap = len(stems & test_id_set)
    if overlap:
        image_dirs.append((overlap, directory))
if not image_dirs:
    raise FileNotFoundError("Could not locate the TGS test/images directory")
overlap, test_image_dir = max(image_dirs, key=lambda item: item[0])
if overlap != len(test_ids):
    missing = sorted(test_id_set - {path.stem for path in test_image_dir.glob("*.png")})[:10]
    raise FileNotFoundError(f"Only found {overlap}/{len(test_ids)} test images; missing examples: {missing}")

depth_frame = pd.read_csv(depths_path)
depth_lookup = dict(zip(depth_frame["id"].astype(str), depth_frame["z"]))
rows = []
for image_id in train_ids:
    image_path = train_image_dir / f"{image_id}.png"
    mask_path = train_mask_dir / f"{image_id}.png"
    with Image.open(mask_path) as handle:
        mask = np.asarray(handle.convert("L")) > 127
    if mask.shape != (101, 101):
        raise ValueError(f"Unexpected mask shape for {mask_path}: {mask.shape}")
    coverage = float(mask.mean())
    rows.append({
        "id": image_id,
        "image": str(image_path),
        "mask": str(mask_path),
        "depth": depth_lookup.get(image_id),
        "coverage": coverage,
        "coverage_class": int(np.clip(np.ceil(coverage * 10), 0, 10)),
    })

processed_frame = pd.DataFrame(rows).sort_values("id").reset_index(drop=True)
processed_train_csv = DATASET_DIR / "jiaozi_tgs_train.csv"
processed_frame.to_csv(processed_train_csv, index=False)

LOCAL_DATA_INFO = {
    "benchmark": "tgs_salt_identification",
    "competition": KAGGLE_COMPETITION,
    "train_csv": str(processed_train_csv),
    "image_dir": str(train_image_dir),
    "mask_dir": str(train_mask_dir),
    "test_image_dir": str(test_image_dir),
    "sample_submission": str(sample_path),
    "depths_csv": str(depths_path),
    "image_column": "image",
    "mask_column": "mask",
    "num_classes": 1,
    "metric": "tgs_mean_precision_iou_0.50_0.95",
    "original_height": 101,
    "original_width": 101,
}
print(json.dumps(LOCAL_DATA_INFO, indent=2, ensure_ascii=False))
print("Coverage classes:\n", processed_frame["coverage_class"].value_counts().sort_index().to_string())

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

from analyzer.image_statistics import ImageStatisticsAnalyzer
from features_extraction_api import module1_pipeline
from pipeline import derive_recommended_epochs, merge_modules, run_module4_generation
from recommender import OutcomeMemory, recommend
from retrieval.rag_retrieval import (
    build_all_task_lists,
    build_graph,
    build_vector_index,
    print_results,
    retrieve_top3_hybrid,
)

print("\n[Notebook] Module 1: parsing TGS requirements...")
m1_output = module1_pipeline(QUERY)
if m1_output is None:
    raise RuntimeError("Module 1 failed")

class LocalSegmentationSplit:
    column_names = ["image", "mask"]

    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, key):
        if isinstance(key, str):
            if key not in self.column_names:
                raise KeyError(key)
            return [self[index][key] for index in range(len(self))]
        row = self.frame.iloc[int(key)]
        with Image.open(row["image"]) as handle:
            image = handle.convert("RGB").copy()
        with Image.open(row["mask"]) as handle:
            mask = handle.convert("L").copy()
        return {"image": image, "mask": mask}

print("\n[Notebook] Module 2: analyzing sampled real TGS images and masks...")
m2_report = ImageStatisticsAnalyzer().analyze({"train": LocalSegmentationSplit(processed_frame)})
if m2_report.get("annotation_format") != "segmentation_mask":
    raise RuntimeError(f"Module 2 did not recognize segmentation masks: {m2_report}")

m3_input = merge_modules(m1_output, m2_report)
m3_input["task_type"] = "image_segmentation"
m3_input["evaluation_metric"] = LOCAL_DATA_INFO["metric"]
m3_input["num_classes"] = 1
m3_input.setdefault("constraints", {})["class_imbalance"] = True
m3_input["class_imbalance"] = True
print("\nMerged Module 3 input:")
print(json.dumps({
    "task_type": m3_input.get("task_type"),
    "data_size": m3_input.get("data_size"),
    "num_classes": m3_input.get("num_classes"),
    "evaluation_metric": m3_input.get("evaluation_metric"),
    "annotation_format": m2_report.get("annotation_format"),
}, indent=2))

print("\n[Notebook] Module 3: retrieving segmentation configurations...")
graph = build_graph()
collection = build_vector_index()
recommendations = retrieve_top3_hybrid(m3_input, graph, collection)
print_results(m3_input, recommendations, graph)
try:
    recommendations = recommend(recommendations, m2_report, m3_input, memory=OutcomeMemory())
    print("[Notebook] Recommender re-ranked candidates.")
except Exception as exc:
    print(f"[Notebook] Recommender skipped: {exc}")

task_lists = build_all_task_lists(recommendations, graph, fmt="nl")
if not task_lists:
    raise RuntimeError("Module 3 returned no task lists")
for task_list in task_lists:
    model_config = task_list.get("model_config")
    if not isinstance(model_config, dict):
        continue
    model_config.update({
        "task_type": "image_segmentation",
        "num_classes": 1,
        "train_csv": LOCAL_DATA_INFO["train_csv"],
        "image_dir": LOCAL_DATA_INFO["image_dir"],
        "mask_dir": LOCAL_DATA_INFO["mask_dir"],
        "image_column": LOCAL_DATA_INFO["image_column"],
        "mask_column": LOCAL_DATA_INFO["mask_column"],
        "test_image_dir": LOCAL_DATA_INFO["test_image_dir"],
        "sample_submission": LOCAL_DATA_INFO["sample_submission"],
        "evaluation_metric": LOCAL_DATA_INFO["metric"],
        "metric": LOCAL_DATA_INFO["metric"],
        "metric_name": LOCAL_DATA_INFO["metric"],
        "offline_smoke": not REAL_TRAINING,
        "benchmark_key": LOCAL_DATA_INFO["benchmark"],
        "competition": LOCAL_DATA_INFO["competition"],
        "validation_fraction": 0.20,
        "seed": 42,
    })
    model_config.setdefault(
        "recommended_epochs",
        derive_recommended_epochs(
            m3_input.get("data_size", "medium"),
            model_config.get("finetune_strategy"),
            bool(model_config.get("pretrained_hf_id")),
        ),
    )

print("\nFirst-ranked Jiaozi config:")
print(json.dumps(task_lists[0]["model_config"], indent=2, ensure_ascii=False)[:4000])
if task_lists[0]["model_config"].get("task_type") != "image_segmentation":
    raise RuntimeError("The first-ranked configuration is not image_segmentation")

print("\n[Notebook] Module 4: generating auditable Jiaozi project...")
module4 = run_module4_generation(
    task_lists,
    OUTPUT_DIR,
    skip_smoke=REAL_TRAINING,
    timeout=120,
    llm_provider="qwen",
)
summary_path = OUTPUT_DIR / "module4_summary.json"
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    raise FileNotFoundError(f"Missing {summary_path}; available: {available}")

DATASET = LOCAL_DATA_INFO["train_csv"]
print("Generated project:", OUTPUT_DIR)
print("Training table:", DATASET)


{
  "benchmark": "tgs_salt_identification",
  "competition": "tgs-salt-identification-challenge",
  "train_csv": "/content/data/tgs-salt-identification-challenge/jiaozi_tgs_train.csv",
  "image_dir": "/content/data/tgs-salt-identification-challenge/train/images",
  "mask_dir": "/content/data/tgs-salt-identification-challenge/train/masks",
  "test_image_dir": "/content/data/tgs-salt-identification-challenge/test/images",
  "sample_submission": "/content/data/tgs-salt-identification-challenge/sample_submission.csv",
  "depths_csv": "/content/data/tgs-salt-identification-challenge/depths.csv",
  "image_column": "image",
  "mask_column": "mask",
  "num_classes": 1,
  "metric": "tgs_mean_precision_iou_0.50_0.95",
  "original_height": 101,
  "original_width": 101
}
Coverage classes:
 coverage_class
0     1562
1      576
2      296
3      224
4      186
5      225
6      205
7      174
8      148
9      159
10     245

[Notebook] Module 1: parsing TGS requirements...

[Notebook] Module 2: ana

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Index] Added 18 backbone entries and refreshed 18 total entries.

──────────────────────────────────────────────────────────────────────
Task   : image_segmentation
Input  : size=medium  priority=balanced
Flags  : class_imbalance
Desc   : Binary semantic segmentation for the Kaggle TGS Salt Identification Challenge. Each grayscale seismic image is 101 by 101 pixels and each mask marks salt versus sediment. The official primary metric is mean per-image precision over IoU thresholds 0.50, 0.55, ..., 0.95; checkpoint selection and threshold tuning must use this metric. The foreground is imbalanced and empty masks are valid. Use an encoder-decoder segmentation model, paired image-mask augmentation, and a loss appropriate for binary imbalanced masks. Resource constraint: one Colab T4 or L4 GPU. Use only public, non-gated pretrained weights. The agent must select the recipe, optimizer, learning rate, augmentation, finetuning strategy, and recommended number of epochs. Preserve spatial outpu

In [8]:
summary_path = OUTPUT_DIR / 'module4_summary.json'
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    print(f'Module 4 summary is missing: {summary_path}')
    print('Available files:', available)
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nGenerated files:')
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(path.name)
else:
    print('Output directory was not created.')


{
  "candidates": [
    {
      "backbone": "segformer",
      "finetune_strategy": "full",
      "loss": "dice_loss",
      "optimizer": "adamw",
      "rank": 1,
      "score": 0.85,
      "task_type": "image_segmentation"
    },
    {
      "backbone": "yolo26",
      "finetune_strategy": "full",
      "loss": "bce_dice_loss",
      "optimizer": "adam",
      "rank": 2,
      "score": 0.7,
      "task_type": "image_segmentation"
    },
    {
      "backbone": "dinov2",
      "finetune_strategy": "full",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 3,
      "score": 0.664,
      "task_type": "image_segmentation"
    }
  ],
  "errors": [],
  "generated_files": [
    "README_generated.md",
    "configs.json",
    "evaluate.py",
    "generation_info.json",
    "infer.py",
    "model.py",
    "model_utils.py",
    "module4_summary.json",
    "requirements.txt",
    "run.py",
    "run_experiments.py",
    "smoke_data.py",
    "train.py",
    "utils.py"
  ]

In [9]:
%%writefile /content/jiaozi_generated_pipeline/tgs_real.py
from __future__ import annotations

import argparse
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from PIL import Image, ImageEnhance
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.transforms import functional as TF


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def flatten_config(config: dict) -> dict:
    model_config = config.get("model_config")
    return {**config, **model_config} if isinstance(model_config, dict) else dict(config)


def load_first_config(project_dir: Path) -> dict:
    configs = json.loads((project_dir / "configs.json").read_text(encoding="utf-8"))
    if not isinstance(configs, list) or not configs:
        raise ValueError("configs.json must contain at least one Jiaozi candidate")
    config = flatten_config(configs[0])
    if config.get("task_type") != "image_segmentation":
        raise ValueError(
            "Jiaozi's first candidate is not image_segmentation: "
            f"{config.get('task_type')!r}"
        )
    return config


def requested_backbone_text(config: dict) -> str:
    return " ".join(
        str(config.get(key) or "")
        for key in ("backbone", "pretrained_name", "pretrained_hf_id", "checkpoint")
    ).lower()


def resolve_image_size(config: dict) -> int:
    # The Agent selected Swin-Base patch4/window7, whose public pretrained
    # torchvision weights are defined for 224x224 inputs.
    if "swin" in requested_backbone_text(config):
        return 224
    requested = int(config.get("image_size") or config.get("input_size") or 128)
    requested = min(256, max(96, requested))
    return int(math.ceil(requested / 32.0) * 32)


def resolve_encoder(config: dict) -> tuple[str, str]:
    requested = requested_backbone_text(config)
    if "swin" in requested:
        return "swin_b", requested
    if "resnet50" in requested or "resnet_50" in requested:
        return "resnet50", requested
    if "resnet18" in requested or "resnet_18" in requested:
        return "resnet18", requested
    if "resnet34" in requested or "resnet_34" in requested:
        return "resnet34", requested
    raise ValueError(
        "The first-ranked Agent backbone is not implemented by the real-data adapter: "
        f"{requested!r}. Refusing to silently replace it with another model."
    )


class DecoderBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int) -> None:
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_channels + skip_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class ResNetUNet(nn.Module):
    def __init__(self, encoder_name: str, pretrained: bool = True) -> None:
        super().__init__()
        factory = getattr(models, encoder_name)
        try:
            encoder = factory(weights="DEFAULT" if pretrained else None)
            pretrained_loaded = pretrained
        except Exception as exc:
            print(f"WARNING: pretrained {encoder_name} weights unavailable: {exc}")
            print("Falling back to random encoder initialization.")
            encoder = factory(weights=None)
            pretrained_loaded = False

        self.encoder_name = encoder_name
        self.pretrained_loaded = pretrained_loaded
        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool = encoder.maxpool
        self.layer1 = encoder.layer1
        self.layer2 = encoder.layer2
        self.layer3 = encoder.layer3
        self.layer4 = encoder.layer4

        if encoder_name == "resnet50":
            channels = [64, 256, 512, 1024, 2048]
        else:
            channels = [64, 64, 128, 256, 512]

        self.dec3 = DecoderBlock(channels[4], channels[3], 256)
        self.dec2 = DecoderBlock(256, channels[2], 128)
        self.dec1 = DecoderBlock(128, channels[1], 64)
        self.dec0 = DecoderBlock(64, channels[0], 48)
        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(48, 32, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[-2:]
        x0 = self.stem(x)
        x1 = self.layer1(self.pool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        x = self.dec3(x4, x3)
        x = self.dec2(x, x2)
        x = self.dec1(x, x1)
        x = self.dec0(x, x0)
        x = self.final_up(x)
        if x.shape[-2:] != input_size:
            x = F.interpolate(x, size=input_size, mode="bilinear", align_corners=False)
        return x



class SwinDecoderBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.block(torch.cat([x, skip], dim=1))


class SwinUNet(nn.Module):
    """Agent-selected Swin-Base/ImageNet-22K encoder plus segmentation decoder."""

    model_name = "swin_base_patch4_window7_224.ms_in22k"

    def __init__(self, pretrained: bool = True) -> None:
        super().__init__()
        create_args = {
            "model_name": self.model_name,
            "features_only": True,
            "out_indices": (0, 1, 2, 3),
        }
        try:
            encoder = timm.create_model(pretrained=pretrained, **create_args)
            pretrained_loaded = pretrained
        except Exception as exc:
            raise RuntimeError(
                "The Agent requires public Swin-B/ImageNet-22K pretrained weights, "
                "but they could not be loaded. Refusing to train a different or "
                "randomly initialized backbone."
            ) from exc

        self.pretrained_loaded = pretrained_loaded
        self.encoder = encoder
        channels = list(encoder.feature_info.channels())
        if len(channels) != 4:
            raise RuntimeError(f"Expected four Swin feature stages, got {channels}")
        self.stage_channels = channels
        self.dec3 = SwinDecoderBlock(channels[3], channels[2], 512)
        self.dec2 = SwinDecoderBlock(512, channels[1], 256)
        self.dec1 = SwinDecoderBlock(256, channels[0], 128)
        self.head = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, 1, kernel_size=1),
        )

    @staticmethod
    def nchw(value: torch.Tensor, expected_channels: int) -> torch.Tensor:
        if value.ndim != 4:
            raise RuntimeError(f"Expected a 4-D Swin feature map, got {tuple(value.shape)}")
        if value.shape[1] == expected_channels:
            return value.contiguous()
        if value.shape[-1] == expected_channels:
            return value.permute(0, 3, 1, 2).contiguous()
        raise RuntimeError(
            f"Could not identify channel dimension {expected_channels} in {tuple(value.shape)}"
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[-2:]
        raw_features = self.encoder(x)
        features = [
            self.nchw(value, channels)
            for value, channels in zip(raw_features, self.stage_channels)
        ]
        skip1, skip2, skip3, x = features
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)
        x = self.head(x)
        return F.interpolate(x, size=input_size, mode="bilinear", align_corners=False)

def build_effective_model(config: dict, pretrained: bool = True) -> tuple[nn.Module, dict]:
    encoder_name, requested = resolve_encoder(config)
    if encoder_name == "swin_b":
        model = SwinUNet(pretrained=pretrained)
        effective_architecture = "SwinUNet"
        effective_encoder = "timm_swin_base_patch4_window7_224_ms_in22k"
    else:
        model = ResNetUNet(encoder_name, pretrained=pretrained)
        effective_architecture = "ResNetUNet"
        effective_encoder = encoder_name

    audit = {
        "jiaozi_requested_backbone": config.get("backbone"),
        "jiaozi_requested_pretrained": config.get("pretrained_hf_id") or config.get("pretrained_name"),
        "effective_architecture": effective_architecture,
        "effective_encoder": effective_encoder,
        "pretrained_encoder_loaded": bool(model.pretrained_loaded),
        "silent_backbone_fallback": False,
        "implementation_note": (
            "The real TGS adapter directly executes the Agent-selected Swin-Base family "
            "with the public timm ImageNet-22K pretrained checkpoint and does not replace "
            "it with a ResNet model."
        ),
        "requested_text": requested,
    }
    return model, audit


class TGSDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, image_size: int, training: bool) -> None:
        self.frame = frame.reset_index(drop=True)
        self.image_size = image_size
        self.training = training

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.frame.iloc[index]
        with Image.open(row["image"]) as handle:
            image = handle.convert("RGB")
        with Image.open(row["mask"]) as handle:
            mask = handle.convert("L")

        if self.training and random.random() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)
        if self.training and random.random() < 0.35:
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.85, 1.15))

        image = TF.resize(image, [self.image_size, self.image_size], antialias=True)
        mask = TF.resize(mask, [self.image_size, self.image_size], interpolation=TF.InterpolationMode.NEAREST)
        image_tensor = TF.to_tensor(image)
        image_tensor = TF.normalize(
            image_tensor,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )
        mask_tensor = (TF.pil_to_tensor(mask).float() / 255.0 > 0.5).float()
        return image_tensor, mask_tensor


def make_image_tensor(path: Path, image_size: int) -> torch.Tensor:
    with Image.open(path) as handle:
        image = handle.convert("RGB")
    image = TF.resize(image, [image_size, image_size], antialias=True)
    tensor = TF.to_tensor(image)
    return TF.normalize(
        tensor,
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )


def dice_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    probabilities = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = (probabilities * targets).sum(dims)
    denominator = probabilities.sum(dims) + targets.sum(dims)
    return (1.0 - (2.0 * intersection + 1.0) / (denominator + 1.0)).mean()


def focal_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    probability = torch.exp(-bce)
    return (((1.0 - probability) ** 2.0) * bce).mean()


def build_loss(loss_name: str, pos_weight: float):
    normalized = loss_name.lower()
    weight = torch.tensor([pos_weight], dtype=torch.float32)

    def compute(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=weight.to(logits.device),
        )
        dice = dice_loss(logits, targets)
        if "focal" in normalized:
            return 0.5 * focal_loss(logits, targets) + 0.5 * dice
        if "bce" in normalized and "dice" in normalized:
            return 0.5 * bce + 0.5 * dice
        if "dice" in normalized:
            # Preserve Dice as the Agent-selected objective while adding BCE
            # stabilization for empty and strongly imbalanced TGS masks.
            return 0.5 * bce + 0.5 * dice
        return bce

    return compute


def build_optimizer(model: nn.Module, config: dict) -> torch.optim.Optimizer:
    name = str(config.get("optimizer") or "adamw").lower()
    learning_rate = float(config.get("learning_rate") or config.get("lr") or 3e-4)
    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if "sgd" in name:
        return torch.optim.SGD(parameters, lr=learning_rate, momentum=0.9, weight_decay=1e-4)
    if "rmsprop" in name:
        return torch.optim.RMSprop(parameters, lr=learning_rate, weight_decay=1e-4)
    if name == "adam":
        return torch.optim.Adam(parameters, lr=learning_rate, weight_decay=1e-5)
    return torch.optim.AdamW(parameters, lr=learning_rate, weight_decay=1e-4)


def tgs_score_one(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    true_sum = int(y_true.sum())
    pred_sum = int(y_pred.sum())
    if true_sum == 0 and pred_sum == 0:
        return 1.0
    if true_sum == 0 or pred_sum == 0:
        return 0.0
    intersection = float(np.logical_and(y_true, y_pred).sum())
    union = float(np.logical_or(y_true, y_pred).sum())
    iou = intersection / max(union, 1.0)
    return float(np.mean([iou > threshold for threshold in np.arange(0.5, 1.0, 0.05)]))


def tgs_score_batch(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean([tgs_score_one(t, p) for t, p in zip(y_true, y_pred)]))


def tune_threshold(targets: np.ndarray, probabilities: np.ndarray) -> tuple[float, float]:
    best_threshold = 0.5
    best_score = -1.0
    for threshold in np.arange(0.10, 0.901, 0.05):
        score = tgs_score_batch(targets, probabilities >= threshold)
        if score > best_score:
            best_threshold = float(round(threshold, 2))
            best_score = float(score)
    return best_threshold, best_score


def estimate_pos_weight(frame: pd.DataFrame) -> float:
    positive = 0
    total = 0
    for path in frame["mask"]:
        with Image.open(path) as handle:
            array = np.asarray(handle.convert("L")) > 127
        positive += int(array.sum())
        total += int(array.size)
    negative = total - positive
    return float(np.clip(negative / max(positive, 1), 1.0, 20.0))


def split_frame(frame: pd.DataFrame, seed: int, validation_fraction: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    stratify = frame["coverage_class"] if "coverage_class" in frame.columns else None
    if stratify is not None and stratify.value_counts().min() < 2:
        stratify = None
    train_frame, validation_frame = train_test_split(
        frame,
        test_size=validation_fraction,
        random_state=seed,
        stratify=stratify,
    )
    return train_frame.reset_index(drop=True), validation_frame.reset_index(drop=True)


def collect_validation(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    image_size: int,
) -> tuple[np.ndarray, np.ndarray]:
    probabilities = []
    targets = []
    model.eval()
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            logits = model(images)
            flipped_logits = torch.flip(model(torch.flip(images, dims=[3])), dims=[3])
            probability = 0.5 * (torch.sigmoid(logits) + torch.sigmoid(flipped_logits))
            probability = F.interpolate(probability, size=(101, 101), mode="bilinear", align_corners=False)
            target = F.interpolate(masks, size=(101, 101), mode="nearest")
            probabilities.append(probability[:, 0].cpu().numpy())
            targets.append(target[:, 0].cpu().numpy() >= 0.5)
    return np.concatenate(targets), np.concatenate(probabilities)


def train(args: argparse.Namespace) -> Path:
    project_dir = Path(args.project_dir).resolve()
    config = load_first_config(project_dir)
    frame = pd.read_csv(config["train_csv"])
    required = {"id", "image", "mask", "coverage_class"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"TGS training CSV is missing: {sorted(missing)}")
    if not frame["image"].map(lambda value: Path(value).is_file()).all():
        raise FileNotFoundError("At least one training image referenced by the CSV is missing")
    if not frame["mask"].map(lambda value: Path(value).is_file()).all():
        raise FileNotFoundError("At least one training mask referenced by the CSV is missing")

    seed = int(config.get("seed") or 42)
    validation_fraction = float(config.get("validation_fraction") or 0.20)
    seed_everything(seed)
    train_frame, validation_frame = split_frame(frame, seed, validation_fraction)
    image_size = resolve_image_size(config)
    batch_size = int(config.get("batch_size") or 16)
    resolved_encoder, _ = resolve_encoder(config)
    if resolved_encoder == "swin_b":
        # Full-finetuned Swin-B is much larger than the former ResNet fallback.
        # This cap is stable on Colab GPUs and changes no model-selection logic.
        batch_size = min(batch_size, 4)
    workers = min(4, max(0, (os.cpu_count() or 2) // 2))

    checkpoint_dir = Path(args.checkpoint_dir).resolve()
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_path = checkpoint_dir / "best_model.pt"
    latest_path = checkpoint_dir / "latest_model.pt"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    is_resuming = latest_path.is_file() and not args.fresh
    # A resumed run already contains the complete encoder state, so do not
    # download ImageNet weights again after a Colab reconnect.
    model, architecture_audit = build_effective_model(config, pretrained=not is_resuming)
    model.to(device)
    optimizer = build_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, args.epochs))
    loss_name = str(config.get("loss") or "bce_dice_loss")
    pos_weight = estimate_pos_weight(train_frame)
    criterion = build_loss(loss_name, pos_weight)

    train_loader = DataLoader(
        TGSDataset(train_frame, image_size, training=True),
        batch_size=batch_size,
        shuffle=True,
        num_workers=workers,
        pin_memory=device.type == "cuda",
        persistent_workers=workers > 0,
        drop_last=False,
    )
    validation_loader = DataLoader(
        TGSDataset(validation_frame, image_size, training=False),
        batch_size=batch_size,
        shuffle=False,
        num_workers=workers,
        pin_memory=device.type == "cuda",
        persistent_workers=workers > 0,
    )

    start_epoch = 1
    best_metric = -1.0
    best_threshold = 0.5
    best_epoch = 0
    if is_resuming:
        state = torch.load(latest_path, map_location=device, weights_only=False)
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        scheduler.load_state_dict(state["scheduler_state_dict"])
        start_epoch = int(state["epoch"]) + 1
        best_metric = float(state.get("best_metric", -1.0))
        best_threshold = float(state.get("best_threshold", 0.5))
        best_epoch = int(state.get("best_epoch") or 0)
        print(f"Resuming from epoch {start_epoch} using {latest_path}")

    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    print(json.dumps({
        "device": str(device),
        "train_rows": len(train_frame),
        "validation_rows": len(validation_frame),
        "image_size": image_size,
        "batch_size": batch_size,
        "epochs": args.epochs,
        "loss": loss_name,
        "optimizer": type(optimizer).__name__,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "positive_pixel_weight": pos_weight,
        "architecture_audit": architecture_audit,
    }, indent=2))

    for epoch in range(start_epoch, args.epochs + 1):
        epoch_started = time.perf_counter()
        print(f"[train] epoch {epoch}/{args.epochs} started", flush=True)
        model.train()
        running_loss = 0.0
        for step, (images, masks) in enumerate(train_loader, start=1):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.item())
            if step % 25 == 0 or step == len(train_loader):
                print(
                    f"epoch={epoch}/{args.epochs} step={step}/{len(train_loader)} "
                    f"loss={running_loss / step:.5f}",
                    flush=True,
                )

        targets, probabilities = collect_validation(model, validation_loader, device, image_size)
        threshold, metric = tune_threshold(targets, probabilities)
        scheduler.step()
        improved = metric > best_metric
        if improved:
            best_metric = metric
            best_threshold = threshold
            best_epoch = epoch

        common_state = {
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric": best_metric,
            "best_threshold": best_threshold,
            "metric_name": "TGS mean precision over IoU thresholds 0.50:0.05:0.95",
            "validation": {
                "epoch_metric": metric,
                "epoch_threshold": threshold,
                "rows": len(validation_frame),
                "split_seed": seed,
                "validation_fraction": validation_fraction,
            },
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "config": config,
            "architecture_audit": architecture_audit,
            "image_size": image_size,
            "loss_name": loss_name,
            "train_ids": train_frame["id"].astype(str).tolist(),
            "validation_ids": validation_frame["id"].astype(str).tolist(),
        }
        torch.save(common_state, latest_path)
        if improved:
            torch.save(common_state, best_path)
            print(f"[train] saved new best checkpoint at epoch {epoch}: {best_path}", flush=True)
        elapsed = time.perf_counter() - epoch_started
        print(
            f"[train] epoch {epoch}/{args.epochs} "
            f"loss={running_loss / max(len(train_loader), 1):.6f} "
            f"val_tgs_metric={metric:.6f} threshold={threshold:.2f} "
            f"best_epoch={best_epoch} best={best_metric:.6f} "
            f"improved={improved} time={elapsed:.1f}s",
            flush=True,
        )

    if not best_path.is_file():
        raise RuntimeError("Training ended without producing best_model.pt")
    print("Best checkpoint:", best_path)
    return best_path


def rle_encode(mask: np.ndarray) -> str:
    pixels = np.asarray(mask, dtype=np.uint8).T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(value) for value in runs)


def rle_decode(value: str, height: int = 101, width: int = 101) -> np.ndarray:
    mask = np.zeros(height * width, dtype=np.uint8)
    value = "" if value is None else str(value).strip()
    if value and value.lower() != "nan":
        numbers = [int(part) for part in value.split()]
        starts = np.asarray(numbers[0::2]) - 1
        lengths = np.asarray(numbers[1::2])
        for start, length in zip(starts, lengths):
            mask[start : start + length] = 1
    return mask.reshape((width, height)).T


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--project-dir", default="/content/jiaozi_generated_pipeline")
    parser.add_argument("--checkpoint-dir", required=True)
    parser.add_argument("--epochs", type=int, required=True)
    parser.add_argument("--fresh", action="store_true")
    args = parser.parse_args()
    train(args)


if __name__ == "__main__":
    main()


Writing /content/jiaozi_generated_pipeline/tgs_real.py


In [12]:
# Extend the existing real-data adapter to execute Agent-selected SegFormer.
# This does not change Module 1–4 or configs.json.

from pathlib import Path
import py_compile

SCRIPT_PATH = Path("/content/jiaozi_generated_pipeline/tgs_real.py")

if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {SCRIPT_PATH}. Run the preceding tgs_real.py generation cell first."
    )

source = SCRIPT_PATH.read_text(encoding="utf-8")


def replace_required(text, old, new, label):
    if new in text:
        print(f"{label}: already applied")
        return text

    if old not in text:
        raise RuntimeError(
            f"Could not apply {label}: expected source block was not found."
        )

    print(f"{label}: applied")
    return text.replace(old, new, 1)


# 1. Add Hugging Face SegFormer imports.
source = replace_required(
    source,
    "from torchvision.transforms import functional as TF\n",
    """from torchvision.transforms import functional as TF
from transformers import SegformerConfig, SegformerForSemanticSegmentation
""",
    "SegFormer imports",
)


# 2. Resolve the image size dynamically.
old_image_size = '''def resolve_image_size(config: dict) -> int:
    # The Agent selected Swin-Base patch4/window7, whose public pretrained
    # torchvision weights are defined for 224x224 inputs.
    if "swin" in requested_backbone_text(config):
        return 224
    requested = int(config.get("image_size") or config.get("input_size") or 128)
    requested = min(256, max(96, requested))
    return int(math.ceil(requested / 32.0) * 32)
'''

new_image_size = '''def resolve_image_size(config: dict) -> int:
    requested_text = requested_backbone_text(config)

    if "swin" in requested_text:
        return 224

    if "segformer" in requested_text:
        requested = int(
            config.get("image_size")
            or config.get("input_size")
            or 224
        )
        requested = min(512, max(128, requested))
        return int(math.ceil(requested / 32.0) * 32)

    requested = int(
        config.get("image_size")
        or config.get("input_size")
        or 128
    )
    requested = min(256, max(96, requested))
    return int(math.ceil(requested / 32.0) * 32)
'''

source = replace_required(
    source,
    old_image_size,
    new_image_size,
    "Dynamic image-size resolution",
)


# 3. Recognize SegFormer without mapping it to another model.
old_encoder_start = '''def resolve_encoder(config: dict) -> tuple[str, str]:
    requested = requested_backbone_text(config)
    if "swin" in requested:
        return "swin_b", requested
'''

new_encoder_start = '''def resolve_encoder(config: dict) -> tuple[str, str]:
    requested = requested_backbone_text(config)

    if "segformer" in requested:
        return "segformer", requested

    if "swin" in requested:
        return "swin_b", requested
'''

source = replace_required(
    source,
    old_encoder_start,
    new_encoder_start,
    "SegFormer encoder resolution",
)


# 4. Insert an actual SegFormer binary-segmentation model.
segformer_class = '''
def resolve_segformer_model_id(config: dict) -> str:
    for key in (
        "pretrained_hf_id",
        "pretrained_name",
        "checkpoint",
        "backbone",
    ):
        value = str(config.get(key) or "").strip()
        if "segformer" in value.lower() and "/" in value:
            return value

    return "nvidia/segformer-b2-finetuned-ade-512-512"


class SegFormerBinarySegmentation(nn.Module):
    """Actual Agent-selected SegFormer with a one-channel salt-mask head."""

    def __init__(
        self,
        config: dict,
        pretrained: bool = True,
    ) -> None:
        super().__init__()

        self.model_id = resolve_segformer_model_id(config)

        if pretrained:
            try:
                network = SegformerForSemanticSegmentation.from_pretrained(
                    self.model_id,
                    num_labels=1,
                    id2label={0: "salt"},
                    label2id={"salt": 0},
                    ignore_mismatched_sizes=True,
                )
            except Exception as exc:
                raise RuntimeError(
                    "The Agent selected SegFormer, but its public "
                    f"pretrained checkpoint could not be loaded: {self.model_id}"
                ) from exc
        else:
            try:
                hf_config = SegformerConfig.from_pretrained(
                    self.model_id,
                    num_labels=1,
                    id2label={0: "salt"},
                    label2id={"salt": 0},
                )
            except Exception as exc:
                raise RuntimeError(
                    "Could not load the SegFormer architecture configuration "
                    f"for checkpoint restoration: {self.model_id}"
                ) from exc

            network = SegformerForSemanticSegmentation(hf_config)

        self.network = network
        self.pretrained_loaded = bool(pretrained)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[-2:]

        outputs = self.network(
            pixel_values=x,
            return_dict=True,
        )

        logits = outputs.logits

        if logits.shape[-2:] != input_size:
            logits = F.interpolate(
                logits,
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )

        return logits


'''

build_marker = "def build_effective_model(config: dict, pretrained: bool = True)"

if "class SegFormerBinarySegmentation" not in source:
    if build_marker not in source:
        raise RuntimeError(
            "Could not locate build_effective_model() in tgs_real.py."
        )

    source = source.replace(
        build_marker,
        segformer_class + build_marker,
        1,
    )
    print("SegFormer model implementation: applied")
else:
    print("SegFormer model implementation: already applied")


# 5. Replace model construction with dynamic Agent-choice execution.
old_build = '''def build_effective_model(config: dict, pretrained: bool = True) -> tuple[nn.Module, dict]:
    encoder_name, requested = resolve_encoder(config)
    if encoder_name == "swin_b":
        model = SwinUNet(pretrained=pretrained)
        effective_architecture = "SwinUNet"
        effective_encoder = "timm_swin_base_patch4_window7_224_ms_in22k"
    else:
        model = ResNetUNet(encoder_name, pretrained=pretrained)
        effective_architecture = "ResNetUNet"
        effective_encoder = encoder_name

    audit = {
        "jiaozi_requested_backbone": config.get("backbone"),
        "jiaozi_requested_pretrained": config.get("pretrained_hf_id") or config.get("pretrained_name"),
        "effective_architecture": effective_architecture,
        "effective_encoder": effective_encoder,
        "pretrained_encoder_loaded": bool(model.pretrained_loaded),
        "silent_backbone_fallback": False,
        "implementation_note": (
            "The real TGS adapter directly executes the Agent-selected Swin-Base family "
            "with the public timm ImageNet-22K pretrained checkpoint and does not replace "
            "it with a ResNet model."
        ),
        "requested_text": requested,
    }
    return model, audit
'''

new_build = '''def build_effective_model(config: dict, pretrained: bool = True) -> tuple[nn.Module, dict]:
    encoder_name, requested = resolve_encoder(config)

    if encoder_name == "segformer":
        model = SegFormerBinarySegmentation(
            config=config,
            pretrained=pretrained,
        )
        effective_architecture = "SegFormerForSemanticSegmentation"
        effective_encoder = model.model_id

    elif encoder_name == "swin_b":
        model = SwinUNet(pretrained=pretrained)
        effective_architecture = "SwinUNet"
        effective_encoder = "timm_swin_base_patch4_window7_224_ms_in22k"

    else:
        model = ResNetUNet(
            encoder_name,
            pretrained=pretrained,
        )
        effective_architecture = "ResNetUNet"
        effective_encoder = encoder_name

    audit = {
        "jiaozi_requested_backbone": config.get("backbone"),
        "jiaozi_requested_pretrained": (
            config.get("pretrained_hf_id")
            or config.get("pretrained_name")
        ),
        "effective_architecture": effective_architecture,
        "effective_encoder": effective_encoder,
        "pretrained_encoder_loaded": bool(model.pretrained_loaded),
        "silent_backbone_fallback": False,
        "implementation_note": (
            "The real TGS adapter directly executes the first-ranked "
            "Agent backbone without silently replacing it with another model."
        ),
        "requested_text": requested,
    }

    return model, audit
'''

source = replace_required(
    source,
    old_build,
    new_build,
    "Dynamic Agent-selected model construction",
)


# 6. Use a safe batch-size cap for large transformer segmenters.
source = replace_required(
    source,
    '''    if resolved_encoder == "swin_b":
        # Full-finetuned Swin-B is much larger than the former ResNet fallback.
        # This cap is stable on Colab GPUs and changes no model-selection logic.
        batch_size = min(batch_size, 4)
''',
    '''    if resolved_encoder in {"swin_b", "segformer"}:
        # Transformer segmentation models require a conservative Colab batch.
        # This does not alter which Agent-recommended model is executed.
        batch_size = min(batch_size, 4)
''',
    "Transformer batch-size safety",
)


# Save and compile-check before any training begins.
SCRIPT_PATH.write_text(source, encoding="utf-8")

py_compile.compile(
    str(SCRIPT_PATH),
    doraise=True,
)

print("\nAdapter syntax validation: PASSED")
print("Adapter file:", SCRIPT_PATH)
print("Supported dynamic choices: SegFormer, Swin, ResNet18/34/50")
print("Silent model fallback: disabled")


SegFormer imports: applied
Dynamic image-size resolution: applied
SegFormer encoder resolution: applied
SegFormer model implementation: applied
Dynamic Agent-selected model construction: applied
Transformer batch-size safety: applied

Adapter syntax validation: PASSED
Adapter file: /content/jiaozi_generated_pipeline/tgs_real.py
Supported dynamic choices: SegFormer, Swin, ResNet18/34/50
Silent model fallback: disabled


## Train the first-ranked Jiaozi recipe on real TGS masks

This keeps the original Jiaozi Module 1–4 flow but now directly executes the selected
Swin Transformer family instead of silently mapping it to ResNet34. The real adapter uses
a Swin-Base encoder, a semantic-segmentation decoder, Dice+BCE stabilization, AdamW,
coverage-stratified 80/20 validation, official TGS checkpoint selection, horizontal-flip
TTA, Drive checkpoints, resume support, and live batch/epoch progress.


In [13]:
# Fresh G4 training of the current first-ranked Agent recommendation.
# Output: one completed-epoch line and best-checkpoint updates.

import hashlib
import json
import os
import re
import subprocess
import sys
from collections import deque
from pathlib import Path

import torch

OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")
TRAIN_SCRIPT = OUTPUT_DIR / "tgs_real.py"
CONFIG_PATH = OUTPUT_DIR / "configs.json"

TOTAL_EPOCHS_FALLBACK = 20

if not TRAIN_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Missing training adapter: {TRAIN_SCRIPT}"
    )

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        f"Missing Agent configurations: {CONFIG_PATH}"
    )

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError(
        "Google Drive is not mounted."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is unavailable. Select the G4 runtime first."
    )

configs = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

if not isinstance(configs, list) or not configs:
    raise RuntimeError(
        "configs.json contains no Agent candidates."
    )

first_candidate = configs[0]
model_config = first_candidate.get(
    "model_config",
    first_candidate,
)

if not isinstance(model_config, dict):
    raise RuntimeError(
        "The first Agent candidate has no valid model_config."
    )

if model_config.get("task_type") != "image_segmentation":
    raise RuntimeError(
        "The first Agent candidate is not image_segmentation."
    )

backbone = str(
    model_config.get("backbone")
    or first_candidate.get("backbone")
    or "unknown"
)

pretrained_id = str(
    model_config.get("pretrained_hf_id")
    or model_config.get("pretrained_name")
    or model_config.get("checkpoint")
    or ""
)

loss_name = str(
    model_config.get("loss")
    or first_candidate.get("loss")
    or "bce_dice_loss"
)

optimizer_name = str(
    model_config.get("optimizer")
    or first_candidate.get("optimizer")
    or "adamw"
)

total_epochs = int(
    model_config.get("recommended_epochs")
    or first_candidate.get("recommended_epochs")
    or TOTAL_EPOCHS_FALLBACK
)

# Validate the actual selected model before creating a long training job.
if str(OUTPUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUTPUT_DIR))

from tgs_real import (
    resolve_encoder,
    resolve_image_size,
)

resolved_encoder, requested_text = resolve_encoder(
    model_config
)

image_size = resolve_image_size(
    model_config
)

safe_name = re.sub(
    r"[^a-z0-9]+",
    "_",
    backbone.lower(),
).strip("_")[:50]

config_hash = hashlib.sha256(
    json.dumps(
        model_config,
        sort_keys=True,
        default=str,
    ).encode("utf-8")
).hexdigest()[:10]

run_name = (
    f"tgs_agent_{safe_name}_g4_"
    f"{config_hash}"
)

checkpoint_dir = (
    Path("/content/drive/MyDrive/Jiaozi/checkpoints")
    / run_name
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# Make these available to later result and submission cells.
ckpt_dir = checkpoint_dir
epochs = total_epochs

print("Agent candidate rank: 1")
print("Agent-recommended backbone:", backbone)
print("Agent-recommended pretrained model:", pretrained_id or "not specified")
print("Resolved real encoder:", resolved_encoder)
print("Agent-recommended loss:", loss_name)
print("Agent-recommended optimizer:", optimizer_name)
print("Resolved image size:", image_size)
print("GPU:", torch.cuda.get_device_name(0))
print("Total epochs:", total_epochs)
print("Checkpoint directory:", checkpoint_dir)
print("Fresh training: True")
print("Silent model fallback: False")
print()

train_command = [
    "/usr/bin/python3",
    "-u",
    str(TRAIN_SCRIPT),
    "--project-dir",
    str(OUTPUT_DIR),
    "--checkpoint-dir",
    str(checkpoint_dir),
    "--epochs",
    str(total_epochs),
    "--fresh",
]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    train_command,
    cwd=str(OUTPUT_DIR),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

if process.stdout is None:
    raise RuntimeError(
        "Could not capture training output."
    )

recent_output = deque(maxlen=100)

for raw_line in iter(
    process.stdout.readline,
    "",
):
    line = raw_line.rstrip()

    if not line:
        continue

    recent_output.append(line)

    completed_epoch_line = (
        line.startswith("[train] epoch ")
        and " started" not in line
    )

    best_checkpoint_line = line.startswith(
        "[train] saved new best checkpoint"
    )

    if completed_epoch_line or best_checkpoint_line:
        print(line, flush=True)

process.stdout.close()
return_code = process.wait()

if return_code != 0:
    print(
        "\n=== TRAINING FAILED: LAST RAW OUTPUT ==="
    )

    for line in recent_output:
        print(line)

    raise subprocess.CalledProcessError(
        return_code,
        train_command,
    )

best_model = (
    checkpoint_dir
    / "best_model.pt"
)

latest_model = (
    checkpoint_dir
    / "latest_model.pt"
)

print("\n=== TRAINING COMPLETE ===")
print("Actual trained backbone:", backbone)
print("Best checkpoint:", best_model)
print("best_model.pt saved:", best_model.is_file())
print("latest_model.pt saved:", latest_model.is_file())


Agent candidate rank: 1
Agent-recommended backbone: segformer
Agent-recommended pretrained model: nvidia/segformer-b2-finetuned-ade-512-512
Resolved real encoder: segformer
Agent-recommended loss: dice_loss
Agent-recommended optimizer: adamw
Resolved image size: 224
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Total epochs: 20
Checkpoint directory: /content/drive/MyDrive/Jiaozi/checkpoints/tgs_agent_segformer_g4_34a966111c
Fresh training: True
Silent model fallback: False

[train] saved new best checkpoint at epoch 1: /content/drive/MyDrive/Jiaozi/checkpoints/tgs_agent_segformer_g4_34a966111c/best_model.pt
[train] epoch 1/20 loss=0.728952 val_tgs_metric=0.423875 threshold=0.80 best_epoch=1 best=0.423875 improved=True time=38.8s
[train] saved new best checkpoint at epoch 2: /content/drive/MyDrive/Jiaozi/checkpoints/tgs_agent_segformer_g4_34a966111c/best_model.pt
[train] epoch 2/20 loss=0.653178 val_tgs_metric=0.583625 threshold=0.80 best_epoch=2 best=0.583625 improved=True time=23.

In [15]:
from pathlib import Path
import torch

checkpoint_dir = Path(ckpt_dir)
best_path = checkpoint_dir / "best_model.pt"
latest_path = checkpoint_dir / "latest_model.pt"

print("Checkpoint directory:", checkpoint_dir)

loaded_states = {}

for label, path in [
    ("BEST", best_path),
    ("LATEST", latest_path),
]:
    print("\n" + "=" * 70)
    print(f"{label} checkpoint:", path)
    print("exists:", path.is_file())

    if not path.is_file():
        continue

    state = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )
    loaded_states[label] = state

    audit = state.get("architecture_audit", {})
    validation = state.get("validation", {})

    print("saved epoch:", state.get("epoch"))
    print("best epoch:", state.get("best_epoch"))
    print("best metric:", state.get("best_metric"))
    print("best threshold:", state.get("best_threshold"))

    print(
        "requested backbone:",
        audit.get("jiaozi_requested_backbone"),
    )
    print(
        "effective architecture:",
        audit.get("effective_architecture"),
    )
    print(
        "effective encoder:",
        audit.get("effective_encoder"),
    )
    print(
        "silent fallback:",
        audit.get("silent_backbone_fallback"),
    )

    if validation:
        print("validation:", validation)

if "BEST" not in loaded_states:
    raise RuntimeError("best_model.pt was not found.")

best_state = loaded_states["BEST"]
best_audit = best_state.get("architecture_audit", {})

effective_architecture = best_audit.get("effective_architecture")
effective_encoder = best_audit.get("effective_encoder")
silent_fallback = best_audit.get("silent_backbone_fallback")

if not effective_architecture:
    raise RuntimeError(
        "The best checkpoint does not contain architecture information."
    )

if silent_fallback is True:
    raise RuntimeError(
        "The checkpoint reports a silent model fallback."
    )

print("\n" + "=" * 70)
print("FINAL VERDICT")
print("Training completed successfully.")
print("Agent-selected architecture:", effective_architecture)
print("Agent-selected encoder:", effective_encoder)
print("Best epoch:", best_state.get("best_epoch"))
print("Best validation TGS metric:", best_state.get("best_metric"))
print("Best probability threshold:", best_state.get("best_threshold"))
print("Best checkpoint:", best_path)
print("Checkpoint validation: PASSED")


Checkpoint directory: /content/drive/MyDrive/Jiaozi/checkpoints/tgs_agent_segformer_g4_34a966111c

BEST checkpoint: /content/drive/MyDrive/Jiaozi/checkpoints/tgs_agent_segformer_g4_34a966111c/best_model.pt
exists: True
saved epoch: 18
best epoch: 18
best metric: 0.7078749999999999
best threshold: 0.9
requested backbone: segformer
effective architecture: SegFormerForSemanticSegmentation
effective encoder: nvidia/segformer-b2-finetuned-ade-512-512
silent fallback: False
validation: {'epoch_metric': 0.7078749999999999, 'epoch_threshold': 0.9, 'rows': 800, 'split_seed': 42, 'validation_fraction': 0.2}

LATEST checkpoint: /content/drive/MyDrive/Jiaozi/checkpoints/tgs_agent_segformer_g4_34a966111c/latest_model.pt
exists: True
saved epoch: 20
best epoch: 18
best metric: 0.7078749999999999
best threshold: 0.9
requested backbone: segformer
effective architecture: SegFormerForSemanticSegmentation
effective encoder: nvidia/segformer-b2-finetuned-ade-512-512
silent fallback: False
validation: {'ep

## Show the selected result

This reads the saved checkpoint metadata without repeating validation. The architecture
audit must report `SwinUNet` and `silent_backbone_fallback: false`; otherwise the run is
rejected rather than silently relabeled.


In [16]:
import json
import torch
from pathlib import Path

candidates = []
try:
    candidates.append(Path(ckpt_dir) / "best_model.pt")
except NameError:
    pass
candidates.extend([
    OUTPUT_DIR / "tgs_checkpoints" / "best_model.pt",
    OUTPUT_DIR / "checkpoints" / "best_model.pt",
])
best_path = next((path for path in candidates if path.is_file()), None)
if best_path is None:
    raise FileNotFoundError("best_model.pt not found in: " + repr([str(p) for p in candidates]))

checkpoint = torch.load(best_path, map_location="cpu", weights_only=False)
print("=== TGS VALIDATION RESULT ===")
print("checkpoint:", best_path)
print("best_epoch:", checkpoint.get("best_epoch"))
print("best_metric:", checkpoint.get("best_metric"))
print("best_threshold:", checkpoint.get("best_threshold"))
print("metric_name:", checkpoint.get("metric_name"))
print("validation:", json.dumps(checkpoint.get("validation"), indent=2))
print("architecture_audit:", json.dumps(checkpoint.get("architecture_audit"), indent=2))


=== TGS VALIDATION RESULT ===
checkpoint: /content/drive/MyDrive/Jiaozi/checkpoints/tgs_agent_segformer_g4_34a966111c/best_model.pt
best_epoch: 18
best_metric: 0.7078749999999999
best_threshold: 0.9
metric_name: TGS mean precision over IoU thresholds 0.50:0.05:0.95
validation: {
  "epoch_metric": 0.7078749999999999,
  "epoch_threshold": 0.9,
  "rows": 800,
  "split_seed": 42,
  "validation_fraction": 0.2
}
architecture_audit: {
  "jiaozi_requested_backbone": "segformer",
  "jiaozi_requested_pretrained": "nvidia/segformer-b2-finetuned-ade-512-512",
  "effective_architecture": "SegFormerForSemanticSegmentation",
  "effective_encoder": "nvidia/segformer-b2-finetuned-ade-512-512",
  "pretrained_encoder_loaded": true,
  "silent_backbone_fallback": false,
  "implementation_note": "The real TGS adapter directly executes the first-ranked Agent backbone without silently replacing it with another model.",
  "requested_text": "segformer segformer-b2 / ade20k nvidia/segformer-b2-finetuned-ade-512-

In [17]:
# Generate official TGS column-major RLE submission.csv

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

if str(OUTPUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUTPUT_DIR))
from tgs_real import build_effective_model, make_image_tensor, rle_decode, rle_encode

sample_path = Path(LOCAL_DATA_INFO["sample_submission"])
test_image_dir = Path(LOCAL_DATA_INFO["test_image_dir"])
submission_path = OUTPUT_DIR / "submission.csv"
sample = pd.read_csv(sample_path)
if list(sample.columns) != ["id", "rle_mask"]:
    raise ValueError(f"Unexpected sample columns: {list(sample.columns)}")

checkpoint = torch.load(best_path, map_location="cpu", weights_only=False)
config = checkpoint["config"]
image_size = int(checkpoint["image_size"])
threshold = float(checkpoint["best_threshold"])
model, inference_audit = build_effective_model(config, pretrained=False)
incompatible = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
if incompatible.missing_keys or incompatible.unexpected_keys:
    raise RuntimeError(str(incompatible))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()
batch_size = min(int(config.get("inference_batch_size") or 8), 8)
encoded_masks = []
batch = []
batch_ids = []

def flush_batch():
    global batch, batch_ids, encoded_masks
    if not batch:
        return
    images = torch.stack(batch).to(device)
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
        logits = model(images)
        flipped_logits = torch.flip(model(torch.flip(images, dims=[3])), dims=[3])
        probabilities = 0.5 * (torch.sigmoid(logits) + torch.sigmoid(flipped_logits))
        probabilities = F.interpolate(
            probabilities,
            size=(101, 101),
            mode="bilinear",
            align_corners=False,
        )
    masks = (probabilities[:, 0] >= threshold).to(torch.uint8).cpu().numpy()
    encoded_masks.extend(rle_encode(mask) for mask in masks)
    batch = []
    batch_ids = []

test_ids = sample["id"].astype(str).tolist()
for index, image_id in enumerate(test_ids, start=1):
    image_path = test_image_dir / f"{image_id}.png"
    if not image_path.is_file():
        raise FileNotFoundError(image_path)
    batch.append(make_image_tensor(image_path, image_size))
    batch_ids.append(image_id)
    if len(batch) >= batch_size:
        flush_batch()
    if index % 1000 == 0:
        print(f"Predicted {index}/{len(test_ids)}", flush=True)
flush_batch()

if len(encoded_masks) != len(sample):
    raise RuntimeError(f"Predictions {len(encoded_masks)} != sample rows {len(sample)}")
submission = sample[["id"]].copy()
submission["rle_mask"] = encoded_masks
if not submission["rle_mask"].map(lambda value: isinstance(value, str)).all():
    raise TypeError("All rle_mask values must be strings")

# Round-trip test proves the encoder follows the required 1-indexed,
# top-to-bottom then left-to-right (Fortran/column-major) convention.
probe_index = next((i for i, value in enumerate(encoded_masks) if value), 0)
probe = rle_decode(encoded_masks[probe_index], 101, 101)
if probe.shape != (101, 101) or not set(np.unique(probe)).issubset({0, 1}):
    raise RuntimeError("RLE round-trip validation failed")

submission.to_csv(submission_path, index=False)
reloaded = pd.read_csv(submission_path, keep_default_na=False, dtype=str)
if list(reloaded.columns) != ["id", "rle_mask"] or len(reloaded) != len(sample):
    raise RuntimeError("Saved submission does not match Kaggle's required schema")
if reloaded["id"].tolist() != sample["id"].astype(str).tolist():
    raise RuntimeError("Saved submission IDs/order differ from sample_submission.csv")

print("Submission written:", submission_path)
print("Rows:", len(submission))
print("Empty masks:", sum(value == "" for value in encoded_masks))
print("Threshold:", threshold)
print("Inference architecture:", json.dumps(inference_audit, indent=2))
display(submission.head())


[transformers] You passed `num_labels=1` which is incompatible to the `id2label` map of length `150`.


Predicted 1000/18000
Predicted 2000/18000
Predicted 3000/18000
Predicted 4000/18000
Predicted 5000/18000
Predicted 6000/18000
Predicted 7000/18000
Predicted 8000/18000
Predicted 9000/18000
Predicted 10000/18000
Predicted 11000/18000
Predicted 12000/18000
Predicted 13000/18000
Predicted 14000/18000
Predicted 15000/18000
Predicted 16000/18000
Predicted 17000/18000
Predicted 18000/18000
Submission written: /content/jiaozi_generated_pipeline/submission.csv
Rows: 18000
Empty masks: 7381
Threshold: 0.9
Inference architecture: {
  "jiaozi_requested_backbone": "segformer",
  "jiaozi_requested_pretrained": "nvidia/segformer-b2-finetuned-ade-512-512",
  "effective_architecture": "SegFormerForSemanticSegmentation",
  "effective_encoder": "nvidia/segformer-b2-finetuned-ade-512-512",
  "pretrained_encoder_loaded": false,
  "silent_backbone_fallback": false,
  "implementation_note": "The real TGS adapter directly executes the first-ranked Agent backbone without silently replacing it with another mod

,id,rle_mask
0,155410d6fa,1 907 910 98 1011 97 1112 96 1213 95 1314 94 1...
1,78b32781d1,56 46 157 46 257 47 357 48 457 49 557 50 657 5...
2,63db2a476a,7272 1 7372 2 7471 4 7563 13 7662 15 7759 19 7...
3,17bfcdb967,4472 9 4546 4 4554 32 4647 44 4719 7 4748 51 4...
4,7ea0fd3c88,


In [18]:
!kaggle competitions submit \
  -c tgs-salt-identification-challenge \
  -f /content/jiaozi_generated_pipeline/submission.csv \
  -m "Jiaozi Agent TGS Salt real segmentation submission"


100% 4.41M/4.41M [00:01<00:00, 3.71MB/s]
Successfully submitted to TGS Salt Identification Challenge

In [20]:
!kaggle competitions submissions -c tgs-salt-identification-challenge


     ref  fileName        date                        description                                         status                     publicScore  privateScore  
--------  --------------  --------------------------  --------------------------------------------------  -------------------------  -----------  ------------  
54720650  submission.csv  2026-07-15 09:55:25.413000  Jiaozi Agent TGS Salt real segmentation submission  SubmissionStatus.COMPLETE  0.69426      0.72590       
